In [ ]:
!pip install crewai crewai-tools requests python-dotenv geopy yt-dlp

In [2]:
import os
from dotenv import load_dotenv

# Method 1: Load from a .env file (recommended)
# Create a .env file in this directory with: GEMINI_API_KEY=your_key_here
load_dotenv()

# Method 2: Set directly (uncomment and paste your key)
# os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

# Verify the key is set
assert os.getenv("GEMINI_API_KEY"), "❌ GEMINI_API_KEY not found! Set it above."
print("✅ API key loaded successfully!")

✅ API key loaded successfully!


In [5]:
import requests
from crewai.tools import tool
from geopy.geocoders import Nominatim


def _geocode_city(city_name: str) -> dict:
    """Convert a city name to latitude/longitude using Nominatim (free)."""
    geolocator = Nominatim(user_agent="crewai_alexa_assistant")
    location = geolocator.geocode(city_name)
    if location:
        return {
            "lat": location.latitude,
            "lon": location.longitude,
            "display_name": location.address,
        }
    return None


# --- Weather condition code mapping ---
WMO_CODES = {
    0: "Clear sky ☀️", 1: "Mainly clear 🌤️", 2: "Partly cloudy ⛅",
    3: "Overcast ☁️", 45: "Foggy 🌫️", 48: "Depositing rime fog 🌫️",
    51: "Light drizzle 🌧️", 53: "Moderate drizzle 🌧️", 55: "Dense drizzle 🌧️",
    61: "Slight rain 🌧️", 63: "Moderate rain 🌧️", 65: "Heavy rain 🌧️",
    71: "Slight snowfall ❄️", 73: "Moderate snowfall ❄️", 75: "Heavy snowfall ❄️",
    80: "Slight rain showers 🌦️", 81: "Moderate rain showers 🌦️",
    82: "Violent rain showers ⛈️", 95: "Thunderstorm ⛈️",
    96: "Thunderstorm with slight hail ⛈️", 99: "Thunderstorm with heavy hail ⛈️",
}


@tool("Weather Lookup")
def get_weather(city: str) -> str:
    """Get the current weather for a given city name.
    Returns temperature, humidity, wind speed, and weather condition.
    Use this whenever the user asks about weather, temperature, or forecasts.
    Args:
        city: The name of the city to check weather for (e.g. 'London', 'New York', 'Tokyo').
    """
    # Step 1: Geocode the city name
    geo = _geocode_city(city)
    if not geo:
        return f"❌ Could not find location for '{city}'. Please try a different city name."

    # Step 2: Call Open-Meteo API
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": geo["lat"],
        "longitude": geo["lon"],
        "current": "temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m,wind_direction_10m",
        "timezone": "auto",
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        current = data["current"]

        weather_desc = WMO_CODES.get(current["weather_code"], "Unknown")

        return (
            f"🌍 Weather for {geo['display_name']}\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"🌡️  Temperature: {current['temperature_2m']}°C\n"
            f"🤔  Feels Like:  {current['apparent_temperature']}°C\n"
            f"💧  Humidity:    {current['relative_humidity_2m']}%\n"
            f"💨  Wind:        {current['wind_speed_10m']} km/h (dir: {current['wind_direction_10m']}°)\n"
            f"🌤️  Condition:   {weather_desc}\n"
            f"🕐  Time:        {current['time']} ({data['timezone']})"
        )
    except requests.RequestException as e:
        return f"❌ Weather API error: {str(e)}"


# --- Quick test ---
print(get_weather.run(city="London"))

ModuleNotFoundError: No module named 'crewai'

In [ ]:
import yt_dlp


@tool("YouTube Search")
def search_youtube(query: str) -> str:
    """Search YouTube for videos matching a query.
    Returns the top 5 results with title, channel, duration, view count, and URL.
    Use this when the user wants to find videos, tutorials, music, or any YouTube content.
    Args:
        query: The search query for YouTube (e.g. 'python tutorial', 'lofi music', 'cooking recipe').
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "extract_flat": False,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            results = ydl.extract_info(f"ytsearch5:{query}", download=False)

        if not results or "entries" not in results:
            return f"❌ No YouTube results found for '{query}'."

        output_lines = [f"🎬 YouTube Results for: '{query}'\n{'━' * 40}"]

        for i, entry in enumerate(results["entries"], 1):
            if not entry:
                continue

            title = entry.get("title", "Unknown Title")
            channel = entry.get("channel", entry.get("uploader", "Unknown"))
            duration = entry.get("duration", 0)
            views = entry.get("view_count", 0)
            video_id = entry.get("id", "")
            url = f"https://www.youtube.com/watch?v={video_id}"

            # Format duration
            mins, secs = divmod(duration or 0, 60)
            hours, mins = divmod(mins, 60)
            dur_str = f"{hours}:{mins:02d}:{secs:02d}" if hours else f"{mins}:{secs:02d}"

            # Format views
            if views:
                if views >= 1_000_000:
                    view_str = f"{views / 1_000_000:.1f}M views"
                elif views >= 1_000:
                    view_str = f"{views / 1_000:.1f}K views"
                else:
                    view_str = f"{views} views"
            else:
                view_str = "N/A views"

            output_lines.append(
                f"\n{i}. {title}\n"
                f"   📺 Channel:  {channel}\n"
                f"   ⏱️  Duration: {dur_str}\n"
                f"   👁️  Views:    {view_str}\n"
                f"   🔗 URL:      {url}"
            )

        return "\n".join(output_lines)

    except Exception as e:
        return f"❌ YouTube search error: {str(e)}"


# --- Quick test ---
print(search_youtube.run(query="CrewAI tutorial"))

In [ ]:
@tool("Currency Exchange")
def get_exchange_rate(base_currency: str, target_currency: str, amount: float = 1.0) -> str:
    """Convert an amount from one currency to another using live exchange rates.
    Use this when the user asks about currency conversion, exchange rates, or forex.
    Args:
        base_currency: The source currency code (e.g. 'USD', 'EUR', 'GBP', 'INR', 'JPY').
        target_currency: The target currency code (e.g. 'EUR', 'USD', 'GBP', 'INR', 'JPY').
        amount: The amount to convert (default 1.0).
    """
    base = base_currency.upper().strip()
    target = target_currency.upper().strip()

    if base == target:
        return f"💱 {amount} {base} = {amount} {target} (same currency)"

    url = "https://api.frankfurter.dev/v1/latest"
    params = {
        "base": base,
        "symbols": target,
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if "rates" not in data or target not in data["rates"]:
            return (
                f"❌ Could not find exchange rate for {base} → {target}.\n"
                f"Available currencies: Check https://api.frankfurter.dev/v1/currencies"
            )

        rate = data["rates"][target]
        converted = round(amount * rate, 4)

        return (
            f"💱 Currency Exchange\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"📊 Rate:      1 {base} = {rate} {target}\n"
            f"💰 Converted: {amount} {base} = {converted} {target}\n"
            f"📅 Date:      {data.get('date', 'N/A')}\n"
            f"ℹ️  Source:    European Central Bank (via Frankfurter)"
        )

    except requests.RequestException as e:
        return f"❌ Forex API error: {str(e)}"


# --- Quick test ---
print(get_exchange_rate.run(base_currency="USD", target_currency="INR", amount=100))

In [ ]:
@tool("List Currencies")
def list_currencies() -> str:
    """List all supported currencies available for foreign exchange conversion.
    Use this when the user asks what currencies are supported or available.
    """
    try:
        response = requests.get(
            "https://api.frankfurter.dev/v1/currencies", timeout=10
        )
        response.raise_for_status()
        currencies = response.json()

        lines = ["💱 Supported Currencies\n" + "━" * 40]
        for code, name in sorted(currencies.items()):
            lines.append(f"  {code}  →  {name}")
        lines.append(f"\nTotal: {len(currencies)} currencies")

        return "\n".join(lines)

    except requests.RequestException as e:
        return f"❌ Error fetching currencies: {str(e)}"


print(list_currencies.run())

In [ ]:
from crewai import Agent, Task, Crew, LLM

# ──────────────────────────────────────────────
# Configure the LLM
# ──────────────────────────────────────────────
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.3,
)

print("✅ LLM configured:", llm.model)

In [ ]:
# ──────────────────────────────────────────────
# 🌦️ Weather Agent
# ──────────────────────────────────────────────
weather_agent = Agent(
    role="Weather Forecaster",
    goal=(
        "Provide accurate, helpful weather information for any city in the world. "
        "Always use the Weather Lookup tool to get real-time data."
    ),
    backstory=(
        "You are a professional meteorologist AI assistant. You specialize in "
        "interpreting weather data and presenting it in a friendly, easy-to-understand "
        "format. You give practical advice like what to wear or whether to carry an umbrella."
    ),
    tools=[get_weather],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# ──────────────────────────────────────────────
# 🎬 YouTube Search Agent
# ──────────────────────────────────────────────
youtube_agent = Agent(
    role="YouTube Content Finder",
    goal=(
        "Find the most relevant YouTube videos for any search query. "
        "Always use the YouTube Search tool and present results clearly."
    ),
    backstory=(
        "You are an expert media curator who knows how to find the best YouTube "
        "content. You evaluate search results and highlight the most relevant videos, "
        "providing brief summaries of why each video might be useful to the user."
    ),
    tools=[search_youtube],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# ──────────────────────────────────────────────
# 💱 Forex Agent
# ──────────────────────────────────────────────
forex_agent = Agent(
    role="Foreign Exchange Specialist",
    goal=(
        "Provide accurate currency conversion and exchange rate information. "
        "Use the Currency Exchange tool for conversions and List Currencies tool "
        "when users need to know available currencies."
    ),
    backstory=(
        "You are a financial analyst specializing in foreign exchange markets. "
        "You provide precise currency conversions and can explain exchange rate "
        "trends. You always use live data from the European Central Bank."
    ),
    tools=[get_exchange_rate, list_currencies],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# ──────────────────────────────────────────────
# 🧠 Router / General Agent
# ──────────────────────────────────────────────
router_agent = Agent(
    role="Personal AI Assistant (Alexa-like)",
    goal=(
        "You are the user's primary assistant. Understand the user's intent and "
        "delegate to the appropriate specialist. For weather queries, delegate to "
        "the Weather Forecaster. For YouTube searches, delegate to the YouTube "
        "Content Finder. For currency/forex queries, delegate to the Foreign "
        "Exchange Specialist. For general knowledge questions, answer directly "
        "using your own knowledge."
    ),
    backstory=(
        "You are a smart, friendly AI personal assistant — like Alexa, but powered "
        "by a team of specialists. You greet users warmly, understand what they need, "
        "and coordinate with your team to deliver the best possible answer. You always "
        "respond in a conversational, helpful tone."
    ),
    tools=[get_weather, search_youtube, get_exchange_rate, list_currencies],
    llm=llm,
    verbose=True,
    allow_delegation=True,
)

print("✅ All agents created!")
print(f"   • Weather Forecaster      — tools: {[t.name for t in weather_agent.tools]}")
print(f"   • YouTube Content Finder   — tools: {[t.name for t in youtube_agent.tools]}")
print(f"   • Foreign Exchange Specialist — tools: {[t.name for t in forex_agent.tools]}")
print(f"   • Router Agent             — tools: {[t.name for t in router_agent.tools]}")

In [ ]:
def ask_assistant(user_query: str) -> str:
    """
    Send a natural language query to the AI assistant crew.
    The router agent will understand the intent and use the right tools.
    """
    # Define the task based on user input
    assistant_task = Task(
        description=(
            f"The user said: '{user_query}'\n\n"
            f"Understand what the user is asking for and fulfill their request:\n"
            f"- If they ask about WEATHER (temperature, forecast, rain, etc.), "
            f"use the 'Weather Lookup' tool with the city name.\n"
            f"- If they want to SEARCH YOUTUBE (find videos, tutorials, music), "
            f"use the 'YouTube Search' tool with their query.\n"
            f"- If they ask about CURRENCY EXCHANGE (conversion, forex, rates), "
            f"use the 'Currency Exchange' tool with the currencies and amount.\n"
            f"- If they ask about available currencies, use the 'List Currencies' tool.\n"
            f"- For general questions, answer using your own knowledge.\n\n"
            f"Respond in a friendly, conversational tone — like a helpful voice assistant."
        ),
        expected_output=(
            "A clear, friendly, and well-formatted response that directly answers "
            "the user's question. Include all relevant data from the tools used."
        ),
        agent=router_agent,
    )

    # Assemble the crew
    crew = Crew(
        agents=[router_agent, weather_agent, youtube_agent, forex_agent],
        tasks=[assistant_task],
        verbose=True,
    )

    # Execute!
    result = crew.kickoff()
    return result.raw


print("✅ Assistant function ready! Use ask_assistant('your question') to chat.")

### 🌦️ Weather Query

In [ ]:
response = ask_assistant("What's the weather like in Tokyo right now?")
print("\n" + "=" * 60)
print("🤖 ASSISTANT RESPONSE:")
print("=" * 60)
print(response)

In [ ]:
response = ask_assistant("Find me some good Python machine learning tutorials on YouTube")
print("\n" + "=" * 60)
print("🤖 ASSISTANT RESPONSE:")
print("=" * 60)
print(response)

In [ ]:
response = ask_assistant("Convert 500 US dollars to Indian Rupees")
print("\n" + "=" * 60)
print("🤖 ASSISTANT RESPONSE:")
print("=" * 60)
print(response)

In [ ]:
response = ask_assistant(
    "I'm planning a trip to Paris. What's the weather there? "
    "Also, how many Euros will I get for 1000 USD?"
)
print("\n" + "=" * 60)
print("🤖 ASSISTANT RESPONSE:")
print("=" * 60)
print(response)

In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║    🤖 CrewAI Alexa-Like Assistant                       ║")
print("║    Type your question below. Type 'quit' to exit.       ║")
print("╚══════════════════════════════════════════════════════════╝")
print()

while True:
    user_input = input("🎤 You: ").strip()

    if not user_input:
        print("   (Please type a question or 'quit' to exit)\n")
        continue

    if user_input.lower() in ["quit", "exit", "bye", "q"]:
        print("\n👋 Goodbye! Have a great day!")
        break

    print("\n⏳ Thinking...\n")

    try:
        response = ask_assistant(user_input)
        print(f"\n🤖 Assistant: {response}\n")
        print("─" * 60 + "\n")
    except Exception as e:
        print(f"\n❌ Error: {str(e)}\n")
        print("─" * 60 + "\n")

In [ ]:
def ask_weather(city: str) -> str:
    """Dedicated weather crew for detailed forecasts."""
    task = Task(
        description=(
            f"Get the current weather for {city}. Use the Weather Lookup tool. "
            f"Provide a friendly summary including what to wear and any weather warnings."
        ),
        expected_output="A detailed, friendly weather report with practical advice.",
        agent=weather_agent,
    )
    crew = Crew(agents=[weather_agent], tasks=[task], verbose=True)
    return crew.kickoff().raw


def ask_youtube(query: str) -> str:
    """Dedicated YouTube crew for video discovery."""
    task = Task(
        description=(
            f"Search YouTube for: '{query}'. Use the YouTube Search tool. "
            f"Present the top results with brief descriptions of why each video is relevant."
        ),
        expected_output="A curated list of YouTube videos with descriptions and links.",
        agent=youtube_agent,
    )
    crew = Crew(agents=[youtube_agent], tasks=[task], verbose=True)
    return crew.kickoff().raw


def ask_forex(base: str, target: str, amount: float = 1.0) -> str:
    """Dedicated forex crew for currency conversions."""
    task = Task(
        description=(
            f"Convert {amount} {base} to {target}. Use the Currency Exchange tool. "
            f"Provide the conversion result with the current rate."
        ),
        expected_output="A clear currency conversion result with the exchange rate and date.",
        agent=forex_agent,
    )
    crew = Crew(agents=[forex_agent], tasks=[task], verbose=True)
    return crew.kickoff().raw


print("✅ Specialized crew functions ready!")
print("   • ask_weather('city')")
print("   • ask_youtube('query')")
print("   • ask_forex('USD', 'EUR', 100)")

In [ ]:
# Example: Direct weather query
print(ask_weather("Mumbai"))

In [ ]:
# Example: Direct YouTube search
print(ask_youtube("best CrewAI projects 2026"))

In [ ]:
# Example: Direct forex conversion
print(ask_forex("GBP", "JPY", 250))